# zero-grad-set-none composite — cx8: post-step buffer update via copy_, then zero_grad(set_to_none=True)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `buffer-copy_-inplace`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "zero-grad-set-none"
DD_ATOM_IDS = ["buffer-copy_-inplace", "zero-grad-set-none"]
DD_SUBTOPICS = ["PyTorch: in-place buffer copy", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The end-of-step bookkeeping in a hand-rolled optimizer has two distinct, atomic moves:

1. **`buffer-copy_-inplace`** — any per-param state buffer (e.g. an EMA running average, or just the latest gradient cached for diagnostics) is updated by writing INTO the existing buffer tensor: `state_buf.copy_(new_value)`. This keeps the buffer's storage stable so callers holding a reference still see the latest value.
2. **`zero-grad-set-none`** — after the step, clear gradients. The PyTorch-default behaviour is `p.grad = None` (set-to-none), NOT `p.grad.zero_()`. Why: setting to `None` (a) frees the grad tensor's memory, (b) makes the next `backward()` allocate a fresh grad (no risk of `+=` accumulating into a stale buffer), (c) skips a kernel launch per param.

**Anatomy.**
```python
for p in params:
    g = p.grad
    state_buf.copy_(g)            # buffer-copy_-inplace (e.g. cache last grad)
    # ... param update ...
    p.grad = None                 # zero-grad-set-none
```

**Why both atoms together.** The buffer holds state that must persist across steps (carries information forward); the grad must NOT persist across steps (carries stale information forward). Both are post-step moves; they look similar but have opposite intent — keep this, drop that.

### Composite Exercise — post-step buffer update via copy_, then zero_grad(set_to_none=True)

**Atoms exercised together**: `buffer-copy_-inplace`, `zero-grad-set-none`

Implement `cx8_post_step_update(params, last_grad_buffers)`.

For each `(p, buf)` pair drawn from the lists:

1. Cache the current `p.grad` into the existing buffer via `buf.copy_(p.grad)`. The buffer tensor object must NOT be rebound — `last_grad_buffers[i]` keeps the same `id` and `data_ptr`.
2. Clear `p.grad` by setting it to `None` (set-to-none semantics). Do NOT use `p.grad.zero_()` — the test catches that variant.

Return `None`. The test checks:
- Buffer values match the pre-call `p.grad` (so the copy_ ran on the right tensor).
- Buffer object identity / storage unchanged.
- `p.grad is None` after the call.
- A fresh `backward()` afterwards re-allocates `p.grad` (proving set-to-none worked).

In [ ]:
def cx8_post_step_update(params, last_grad_buffers):
    for p, buf in zip(params, last_grad_buffers):
        # Atom A (buffer-copy_-inplace): cache the grad into the existing buffer storage.
        buf.copy_(p.grad)
        # Atom B (zero-grad-set-none): None, NOT p.grad.zero_().
        p.grad = None
    return None


<details><summary>Show solution — cx8</summary>

```python
def cx8_post_step_update(params, last_grad_buffers):
    for p, buf in zip(params, last_grad_buffers):
        # Atom A (buffer-copy_-inplace): cache the grad into the existing buffer storage.
        buf.copy_(p.grad)
        # Atom B (zero-grad-set-none): None, NOT p.grad.zero_().
        p.grad = None
    return None
```

Subtle gotcha: `buf.copy_(p.grad)` runs BEFORE `p.grad = None`. If you swap the order, `p.grad` is `None` at the time of copy_ and you'd hit a TypeError. The set-to-none flavor of zero_grad is the PyTorch default since 1.7 — `p.grad.zero_()` is the older behaviour, kept for backward-compat but slower.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx8',
        'subtopics': ["PyTorch: in-place buffer copy", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()